## Imports and setup

This notebook is in a customized project and requires a custumized python module (.py file) to run. By default the path to the start module for a custom project is ./src/"project_name"; it needs to contain a startup function. The default name of that function is "Run_process". In this project, [AI4SH](https://ai4soilhealth.eu), the path to the custom project is ./src/ai4sh and the first cell calls the Run_process function in that package.

In [ ]:
#  Standard library imports
from os import path, getcwd

import sys 

# Get the working directory of the notebook
notebook_path = getcwd()

# Set the path to the root directory of this jupyter notebook (python) project
module_path = path.abspath(path.join('../..'))

if module_path not in sys.path:
    
    sys.path.append(module_path)

# Package application imports
from src.lib import Initiate_process

# Import the custom class for running processes defined in the ai4sh package
from src.ai4sh import Run_process

## Scheme file
To run this cell you need to set the [scheme_file](https://xspatula.github.io/setup_core_db_docs/framework/scheme_file/), that defines the location (path) to the project and associated process files and your user. You can set the path to the scheme file with a relative path (relative to this notebook), in your home directory using tilde (~) or with an absolute path in the syntax of your operating system. 

In [ ]:
# Set the path to the scheme_file  

scheme_file = '../scheme_ai4sh.json'

# 1 Linear regressors OLS and Huber

The first block compares two linear regressors, Ordinary Least Square (OLS) and Huber, where Huber is robust to outliers.

## 1.1 Comparing the effects of pre-processing

The cells under 1.1 compare OLS and Huber applied to raw (non pre-processed) and preprocessed data.

### 1.1.1 Machine Learning modelling on raw data with train-test

The first modeling example is using the raw compiled data, with the original, not pre-processed spectral data. It only applies train-test modeling, two indicators (c-org and n-tot) and two regressors (ols and huber).


In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_raw_tt.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 1.1.2 Machine Learning modelling on range indicators range selected data with train-test
In cell 1.1.2 the regression instead use the range reduced indicators and raw spectra for modeling. The range of both the x and the y axis should be lower, reduced to approximately the ranges defined in indicator range process.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_range_tt.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 1.1.3 Machine Learning modelling on outlier pre-processed indicators and spectra

The thrid modeling example use the same indicators (c-org and n-tot) and regressors (ols and huber) as cells 1.1.1 and 1.1.2, but with pre-processed data with indicator outliers removed and a spectral pre-processing chain. Again, the range of both the x and the y axis should be lower compared to cell 1.1.1, reduced dependent on the outlier detection process.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_pre-processed_tt.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 2 Regressors with one-hot covariates
The texture classification of each soil sample is automatically calculated from the clay, silt and sand content when an observation is added to the database. In 2.1 soil texture is applied as a one-hot encoded (one 0/1 column per class) covariate. 

To set covariates in a JSON process file, define a `covariate_array` (where `all` = every covariate of the compiled dataset). Numeric covariates are used as they are; nominal (class) covariates, e.g. soil texture, are one-hot encoded (one 0/1 column per class).

- `class_grouping` pools classes before encoding (array of arrays per covariate); every class in the data must be placed in a group. Predefined groupings are separate covariates, e.g. `soil texture usda major group`.
- Samples without a value for every covariate are skipped, both when modelling and when predicting; test samples with a class that has no training sample are also skipped.
- The saved model needs the spectra and the covariates; model and plot names get the suffix `_cv-<covariates>`.
- Tree based regressors (randfor, dectree) use the classes directly; for linear regressors a class shifts the prediction by a fixed amount.

Cells under 2.1 apply simpler regressors (OLS and random forest) starting from PCA compressed data. Cells under 2.2 apply more advanced regressors demanding more input and start from pre-processed spectra without PCA compression. 

## 2.1 Simple regressors

Cells 2.1 apply simpler regressors (OLS and decision tree) with texture as one-hot class covariate and spectral data decomposed to 5 principal components using PCA.

### 2.1.1 Machine Learning modelling on raw with train-test and kfolds with covariates

The third modeling example is using the raw compiled data, with the original, not pre-processed spectral data. It applies both train-test and kfold (5) modeling, two indicators (c-org and n-tot) and two regressors (ols and random forest). The modeling divides the models according to 3 textural classes (fine, medium, coarse - see JSON process file). 

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_texture_raw_tt+kf5.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.1.2 Machine Learning modelling on pre-processed indicators and spectra with train-test and kfolds with covariates

The fourth modeling example use the same indicators and regressors as cell 2.1.1, but on pre-processed indicators and spectra, with the spectra represented as 5 principal components.


In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_texture_pre-processed_tt+kf5.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 2.2 Advanced regressors

Cells 2.2.1–2.2.5 are examples of how to apply more advanced regressors (PLSR, SVR, Cubist, Random Forest and MLP) that require more input data (spectra) and thus use 116 spectral bands instead of 5 components from PCA as input. The dataset is small, or too small, for applying these advanced regressors but are included for reference. The size of each class group becomes too small for some models to converge and warnings are issued.

All cells use the same data: indicators with outliers removed, the soil texture class (3 groups) as covariate and the spectra after SNV, first derivative and mean centring **without PCA** (`neospectra_10nm_olif_texture.snv.d1.mc`, created in cell 3.1.4 of `compile_dataframe.ipynb`). The indicators are c-org, n-tot and clay, with train-test and 5-fold cross validation. The hyper parameters are set in `regressors/regression_model_parameters_tuned.json`, chosen from 5-fold tests on this dataset (144 samples). Differences in RPIQ below about 0.15 are within the noise of a single fold split.

### 2.2.1 PLSR on the full pre-processed spectra with covariates

Cell 2.2.1 applies a Partial Least Square Regression (PLSR).

Things to consider for PLSR:
- **Do not end the spectra pre-processing chain with PCA.** PLSR builds its own components, chosen for the indicator. After a decomposition (PCA) it can only use the saved components and becomes a regression on the PCA scores.
- **`n_components` is the only important hyper parameter.** Too few components underfit, too many fit noise. On this dataset 3–5 components were best, 10 (the `standard` set) was clearly worse and 20 overfitted (RPIQ c-org 1.80 with 5, 1.60 with 10, 1.39 with 20). The best number grows with the number of samples and the complexity of the indicator, so test e.g. 3, 5, 8, 12 when the data change. Use the k-fold result, not train-test, to choose.
- **Rule of thumb:** keep `n_components` well below the number of samples per fold divided by 10.
- **`scale: false`** lets the chain decide the scaling (mean centring here). Use autoscaling in the chain, not `scale: true`, if all bands should weigh the same.
- **Capped automatically.** If `n_components` is more than the data allow, it is reduced to the rank of the centred data and a note is printed, e.g. `n_components 10 → 7 (130 samples, 8 features, rank 7)`.
- Covariates are standardised and added as columns; the one-hot classes of a nominal covariate count as columns, not as components.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_texture_plsr_snv-d1-mc_tt+kf5.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.2.2 Support Vector Regression (SVR) on the full pre-processed spectra with covariates

Cell 2.2.2 applies a Support Vector Regression (SVR).

 A linear kernel with `C: 10` and `epsilon: 0.01` was the best SVR on this dataset and the best model for c-org (RPIQ 2.0). Things to consider:
- `C` sets how hard errors are penalised: higher C fits the training samples closer. Test 1, 10, 100.
- `epsilon` is in the **unit of the indicator**: errors smaller than epsilon are ignored. 0.05 is fine for c-org (%) but too wide for n-tot (values around 0.1–0.4 %). Keep it well below the RMSE of the indicator.
- The `rbf` kernel (C 10) was better for clay (RPIQ 2.9) but worse for c-org and n-tot; try both.
- SVR is not scaled inside the model: the chain (and the standardised covariates) decide the feature scale.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_texture_svr_snv-d1-mc_tt+kf5.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.2.3 Cubist on the full pre-processed spectra with covariates

Cubist builds rule-based linear models. Things to consider:
- `n_committees: 10` (boosting-like ensemble of rule sets) was clearly better than the default 1 (RPIQ c-org 1.66 vs 1.31).
- `neighbors` (1–9) adjusts the predictions with the nearest training samples; 5 gave no improvement here.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_texture_cubist_snv-d1-mc_tt+kf5.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.2.4 Random forest on the full pre-processed spectra with covariates

Random forest is the most stable model across the indicators. Things to consider:
- `n_estimators: 300` instead of 30: more trees give more stable predictions, at the cost of time.
- `max_features: 0.33` (a third of the features at each split) is the usual choice for regression and was better than all features.
- Trees do not extrapolate: predictions stay within the range of the training samples.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_texture_randfor_snv-d1-mc_tt+kf5.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 2.2.5 MLP neural network on the full pre-processed spectra with covariates

Multi Layer  Perceptron (MLP) is a neural network type of regressor and applied in cell 2.2.5. 

With 144 samples a neural network is not recommended: it was the weakest and least stable model in the tests (n-tot RPIQ between 0.25 and 1.7 depending on the settings). If you use it:
- keep the network small (`hidden_layer_sizes: [20]`) and the regularisation strong (`alpha: 10`);
- the default `[100, 100]` has more weights than there are samples many times over;
- the features are standardised inside the model (StandardScaler).

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_texture_mlp_snv-d1-mc_tt+kf5.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 3 Indicator array



## 3.1 example

### 3.1.1 All indicators of the dataframe, one regressor, custom plot layout

Cell 3.1.1 models every indicator of the compiled dataframe `neospectra_10nm_olif_texture.snv.d1.mc` (`indicator_array`: cec, clay, ca2+, ec, n-tot, c-org, ph-water) with PLSR, the texture covariate and 5-fold cross validation only. Leave `indicator_array` out (or empty) to model every indicator of the dataframe.

The plots are arranged by a custom plot layout, `plot_layout/scatter_kf_4-per-row.json` (path relative to the project root), instead of a preset: k-fold scatter only, 4 per row and slightly smaller panels. With one regressor the indicators flow 4 per row, so the 7 indicators fill two rows. `save_plot_layout: true` also saves the grid under `plot/<dataframe>_grid/`.

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_all-indicators_plsr_snv-d1-mc_kf5.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 4 Coveriate importance

Map the importance of the individual spectral bands (covaraites) for combinations of a particular indicator and a regressor. 

## 4.1 Evaluate the importande of Principal Components in linear models

### 4.1.1 Importance of the PCA components in OLS and Huber, without the model tests

Relative importance of the five PCA components (`neospectra_olif_10nm.snv.d1.mc.pca5`) for estimating c-org and n-tot with OLS and Huber.

`importance_only: true` fits the regressors and maps the importance only: no scatter plots, metrics, k-fold predictions or saved models. With train-test the regressors are fitted on the training samples and the permutation importance is computed on the test samples. Two kinds of importance are shown (both regressors are linear):
- **feature importance**: the absolute coefficient of each component (for random forest/decision tree the impurity decrease, for PLSR the loadings; knn, mlp and cubist have none);
- **permutation importance**: how much r² drops when the values of a component are shuffled (all regressors except cubist).

The custom plot layout `plot_layout/importance_pca.json` puts the importance and permutation panels of both regressors on one row per indicator (`importance_width: 1`, as there are only 5 components).

In [ ]:
#%%script false --no-raise-error
process_file = 'project_neospectra/process/ml_model/regression_modeling_importance_only_pca5_tt.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)